# Henze et al. (2000) on CRCNS hc-1

Paper: Henze, Borhegyi, Csicsvari, Mamiya, Harris and Buzsaki (2000), "Intracellular features
predicted by extracellular recordings in the hippocampus in vivo", J Neurophysiol 84:390-400,
doi:10.1152/jn.2000.84.1.390. The claims file `claims/henze-2000.yaml` was written from the
version of record (Introduction, Methods, Results, Discussion and the legends of Figures 1 to 8;
the paper has no supplement and links no code).

Dataset: CRCNS hc-1, "Simultaneous intracellular and extracellular recordings in the
hippocampus of anesthetized rats", doi:10.6080/K02Z13FP, read anonymously from the public S3
bucket `crcnsarchive` under `hc-1/live/`. CRCNS data sets are not versioned
(docs/archives.md), so the version is recorded as `unversioned`, pinned by the MD5 sums of the
data set's `checksums.md5`, which the next cell prints for every file read. The `live` folder is
the only copy of the data; `hc-1/orig/ver_1_0/` holds only the documentation of version 1.0.
The data descriptor read is version 1.1 (April 21, 2021), which adds only a note that the
recording times in IntraExtra.xls are wrong in many cases. So the published files under
`live/` are the version used: there is no other.

| Claim | Tested or blocked |
|---|---|
| sample-cells | tested |
| fig1c-start-end-timing | tested |
| ap-duration-match | tested |
| duration-correlation | tested |
| neg-peak-precedes-intra-peak | tested |
| fig1c-derivative-mismatch | tested |
| notch-and-ahp-landmarks | tested |
| fig2-interneuron-narrow | blocked: the basket cell is not identified among the tetrode cells |
| fig3b-latency-vs-amplitude | tested |
| fig3a-example-pair | tested on the candidate pairs the folder names allow |
| fig3c-backprop-velocity | tested |
| fig4bc-burst-attenuation | tested |
| fig4d-amplitude-correlation | tested |
| fig5-theta-smaller-spikes | blocked: behaving-animal recordings not in the data set |
| theta-higher-rates | blocked: behaving-animal recordings not in the data set |
| fig6-amplitude-vs-distance | blocked: no tetrode-to-soma distances in the data set |
| tetrode-cell-count-estimate | blocked: depends on fig6-amplitude-vs-distance |
| silent-cells-inference | blocked: depends on tetrode-cell-count-estimate |
| fig7-vertical-spread | tested |
| propagation-velocity-range | tested on the anesthetized silicon-probe recordings |
| fig8-neighbouring-shanks | blocked: behaving-animal recordings not in the data set |

In [ ]:
import io
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import hc1

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
DATASET = "crcns:hc-1/unversioned"
GRID = hc1.GRID

listing = hc1.list_data()
md5 = dict(reversed(l.split()) for l in hc1.read_small("checksums.md5").decode().splitlines() if l.strip())
print("files under hc-1/:", len(listing))
for k, n in listing:
    rel = k.replace("hc-1/live/", "")
    if n:
        print(f"{k:60s} {n:>12d}  md5 {md5.get(rel, '-')}")

## Documentation and the spreadsheet

The README of the data set and IntraExtra.xls, the per-cell spreadsheet the README points to.

In [ ]:
readme = hc1.read_small("README.txt").decode("latin1")
print(readme[:400])
sheet = hc1.read_spreadsheet()
sheet = sheet.dropna(how="all")
print(sheet.columns.tolist())
print(sheet.to_string())

## Load

Every .dat file of every cell folder is streamed once, from the zip in place. In each file the
intracellular channel is chosen from the data, action potentials are detected on it, and
windows of the intracellular and extracellular traces around each AP peak are averaged and
measured (`hc1.py` says where each parameter comes from). This takes about 5 minutes with three
folders at a time; peak memory per worker is about 1.6 GB.

In [ ]:
t0 = time.time()
R = hc1.load_all(workers=3)
print(f"{len(R)} folders processed in {time.time() - t0:.0f} s")

## Data check

Per file: format, sampling rate, channel count, the channel chosen as intracellular, its median
potential outside current steps (with the scale in `hc1.scales`), the number of APs, and the
fraction of time in current steps.

In [ ]:
rows = []
for fo, r in R.items():
    for f in r["files"]:
        rows.append(dict(folder=fo, member=f["member"], xml=f["xml"], bits=12 if f["twelve"] else 16,
                         sr=f["sr"], nc=f["nc"], ec=",".join(map(str, f["ec"])), ic=f["ic"],
                         dur_s=round(f["n_samples"] / f["sr"]), n_ap=f["n_spikes"],
                         vrest=round(f.get("vrest_median", np.nan), 1),
                         in_step=round(f.get("frac_in_step", np.nan), 2)))
F = pd.DataFrame(rows)
print(F.to_string())
print()
print("cell folders:", F.folder.nunique(), " files:", len(F), " files with APs:", int((F.n_ap > 0).sum()),
      " files without an intracellular channel found:", int(F.ic.isna().sum()))
print("files by format and sampling rate:")
print(F.groupby(["bits", "sr"]).size().to_string())
print("files without an xml (14 channels at 20 kHz assumed):", F[~F.xml].member.tolist())
print("folders with a README:", sorted({fo for fo, r in R.items() if r["files"] and r["files"][0]["readme"]}))
for fo, r in R.items():
    if r["files"] and r["files"][0]["readme"]:
        print("  ", fo, r["files"][0]["readme"].replace("\n", "; "))
print("APs detected in total:", int(F.n_ap.sum()))

Scale check of the intracellular channel. The spreadsheet gives a resting potential (`Vrest`)
for 18 of the 39 cells; the median potential measured here, with 20 V / 2^bits / 50 per step,
is set beside it.

In [ ]:
sheet["folder"] = sheet["cell"].str.lower()
vr = F[F.n_ap > 0].groupby("folder").vrest.median()
chk = sheet[["folder", "Vrest", "offset(mV)", "dend?", "Interneuron", "Electrode Type", "# of files", "recording time"]].copy()
chk["vrest_measured"] = chk.folder.map(vr)
chk["files_in_zip"] = chk.folder.map(F.groupby("folder").size())
chk["minutes_in_zip"] = chk.folder.map(F.groupby("folder").dur_s.sum() / 60).round(1)
print(chk.to_string())
num = pd.to_numeric(chk.Vrest, errors="coerce")
d = (chk.vrest_measured - num).dropna()
print(f"measured minus spreadsheet Vrest over {len(d)} cells: median {d.median():.1f} mV, "
      f"within 5 mV for {int((d.abs() <= 5).sum())}, range {d.min():.1f} to {d.max():.1f} mV")
print("folders not in the spreadsheet:", sorted(set(R) - set(sheet.folder)))

Recordings deposited more than once. Within each folder, every pair of files is compared by
their AP trains: the fraction of the shorter train's APs that fall within 1 ms of an AP of the
other after the best constant offset. A fraction near 1 means the same recording.

In [ ]:
dups = []
for fo, r in R.items():
    fs = [f for f in r["files"] if f["n_spikes"] >= 20]
    for i in range(len(fs)):
        for j in range(i + 1, len(fs)):
            off, frac = hc1.align_trains(fs[i]["spikes"]["time_s"], fs[j]["spikes"]["time_s"])
            dups.append(dict(folder=fo, a=fs[i]["member"], b=fs[j]["member"], na=fs[i]["n_spikes"],
                             nb=fs[j]["n_spikes"], offset_s=round(off, 3), matched=round(frac, 2)))
D = pd.DataFrame(dups)
print(f"{len(D)} pairs compared; matched fraction: median {D.matched.median():.2f}, "
      f"95th percentile {D.matched.quantile(0.95):.2f}")
print(D[D.matched >= 0.3].to_string())
# a pair matched at 0.7 or more is one recording; the shorter file is dropped
DROP = set()
for _, p in D[D.matched >= 0.7].iterrows():
    DROP.add(p.a if p.na < p.nb else p.b)
print("dropped as duplicates:", sorted(DROP))

Clock check. The extracellular and intracellular channels of a file are columns of one sampled
matrix, so they share a clock by construction. As a contrast, the averaged extracellular trace
at the AP times is set against the same average 25 ms after each AP: a spike locked to the
intracellular AP should appear only at the true times.

In [ ]:
clock = []
for fo, r in R.items():
    w = hc1.cell_waves(r["files"], "all", DROP)
    ws = hc1.cell_waves(r["files"], "shift", DROP)
    if w is None:
        continue
    a = np.array([hc1.ec_measures(w["ec"][j], w["noise"][j])["amp_neg"] for j in range(len(w["ec"]))])
    b = np.array([hc1.ec_measures(ws["ec"][j], ws["noise"][j])["amp_neg"] for j in range(len(ws["ec"]))])
    j = int(np.argmax(a))
    clock.append(dict(folder=fo, n=w["n"], channel=j, amp_at_ap_uV=round(a[j], 1), snr_at_ap=round(a[j] / w["noise"][j], 1),
                      amp_25ms_later_uV=round(b[j], 1), snr_25ms_later=round(b[j] / ws["noise"][j], 1)))
C = pd.DataFrame(clock)
print(C.to_string())
print(f"cells with an averaged spike above 2 SD at the AP times: {(C.snr_at_ap > 2).sum()} of {len(C)}; "
      f"25 ms later: {(C.snr_25ms_later > 2).sum()}; median SNR at AP {C.snr_at_ap.median():.1f}, "
      f"25 ms later {C.snr_25ms_later.median():.1f}")

## The cells

One row per cell folder. Electrode: the spreadsheet gives "tetrode" for D145 and D149 and the
2x6 silicon probe from D157 on; folders from D53 to D151 have four (D81 to D83: eight)
extracellular channels, a wire tetrode, and those from D156 on twelve or six, the silicon probe.
Resting potential: median over the cell's files with APs. Kept: resting potential more
negative than -55 mV (Methods and Results). Somatic or dendritic, from the paper (Results, "The
recordings were determined to be from the dendrites based on the lack of a fast
afterhyperpolarization and/or an action potential peak value <0 mV"): dendritic when the peak
of the averaged AP is below 0 mV or it has no fast AHP (`hc1.ic_measures`), somatic otherwise.
The extracellular spike is present when its averaged negative peak exceeds 2 SD of the averaged
noise on the largest channel (claims file, Fig. 6 and Fig. 7 criteria).

In [ ]:
def cell_row(fo, r):
    w = hc1.cell_waves(r["files"], "all", DROP)
    files = [f for f in r["files"] if f["n_spikes"] > 0 and f["member"] not in DROP]
    out = dict(folder=fo, animal=hc1.animal(fo), electrode="silicon probe" if hc1.animal(fo) >= 156 else "tetrode")
    if w is None:
        out.update(n_ap=0)
        return out, None
    im = hc1.ic_measures(w["ic"])
    amps = [hc1.ec_measures(w["ec"][j], w["noise"][j])["amp_neg"] for j in range(len(w["ec"]))]
    best = int(np.argmax(amps))
    em = hc1.ec_measures(w["ec"][best], w["noise"][best])
    out.update(n_files=len(files), n_ap=w["n"], vrest=np.median([f["vrest_median"] for f in files]),
               ap_peak=im["peak"], ap_amp=im["amp_thr_a"], fahp=im["fahp"], best=best, n_ec=len(w["ec"]),
               ec_neg_uV=em["amp_neg"], ec_snr=em["amp_neg"] / w["noise"][best], ec_np_uV=em["amp_np"],
               initial_pos_uV=w["ec"][best][(GRID >= -1) & (GRID <= em["t_neg"])].max() - em["base"])
    out["kept"] = out["vrest"] < -55
    out["site"] = "somatic" if (im["peak"] >= 0 and im["fahp"]) else "dendritic"
    out["site_peak_only"] = "somatic" if im["peak"] >= 0 else "dendritic"
    out["ec_present"] = out["ec_snr"] > 2
    return out, w

cells, WAV = [], {}
for fo, r in R.items():
    row, w = cell_row(fo, r)
    cells.append(row)
    WAV[fo] = w
T = pd.DataFrame(cells).set_index("folder")
T["kept"] = (T.vrest < -55).astype(bool)
T["ec_present"] = (T.ec_snr > 2).astype(bool)
print(T.round(2).to_string())
print()
print(T.groupby(["electrode", "kept", "site"]).size().to_string())
lab = sheet.set_index("folder")
T["sheet_dend"] = lab["dend?"].reindex(T.index)
T["sheet_interneuron"] = lab["Interneuron"].reindex(T.index)
cmp = T.loc[T.index.isin(lab.index) & T.ap_peak.notna(), ["site", "sheet_dend", "sheet_interneuron", "ap_peak", "fahp"]]
print("\nsomatic or dendritic here against the spreadsheet's dend? column (y = dendritic):")
print(cmp.to_string())
agree = ((cmp.site == "dendritic") == (cmp.sheet_dend == "y"))
print(f"agree for {int(agree.sum())} of {len(cmp)} cells")

## sample-cells

> Of 33 neurons recorded both intracellularly and by the tetrode, 22 were kept for analysis
> (stable recording, resting potential more negative than -55 mV, satisfactory labelling), 21
> pyramidal cells and one interneuron.

Criterion: counts of distinct cells with tetrode recordings (single-wire cases apart); kept:
resting potential more negative than -55 mV, and the data set's cell type labels where it has
them. Each folder is one cell (README: "Each folder corresponds to an experiement with one
cell"). No folder has a single extracellular channel, so no single-wire case is found.

In [ ]:
tet = T[T.electrode == "tetrode"]
n_total = len(tet)
n_with_ap = int((tet.n_ap > 0).sum())
n_kept = int(tet.kept.sum())
in_lab = sheet.set_index("folder")
tet_int = [f for f in tet.index if f in in_lab.index and in_lab.loc[f, "Interneuron"] == "y"]
print(f"tetrode cell folders: {n_total} (with APs found: {n_with_ap}); expected 33 +/- 3.3")
print(f"kept by resting potential < -55 mV: {n_kept}; expected 22 +/- 2.2")
print(f"tetrode cells in the spreadsheet: {sum(f in in_lab.index for f in tet.index)}; labelled interneurons among them: {len(tet_int)}")
print(f"pyramidal (kept, no interneuron label): {n_kept - len(tet_int)}; expected 21 +/- 2.1; interneurons: {len(tet_int)}, expected 1")
print(f"all cell folders, any electrode: {len(T)}; kept: {int(T.kept.sum())}; "
      f"labelled interneurons: {int((sheet.Interneuron == 'y').sum())} ({sheet.folder[sheet.Interneuron == 'y'].tolist()})")
print("tetrode cells by resting potential (mV):", tet.vrest.round(1).sort_values().to_dict())

**Result: not reproduced.** The data set holds 21 tetrode cells, outside 33 +/- 3.3. All 21 have
a resting potential below -55 mV, so 21 are kept (22 +/- 2.2 expected, within tolerance). No
tetrode cell carries a cell type label (2 of the 21 are in the spreadsheet, neither labelled an
interneuron), so the interneuron count from labels is 0 where 1 is expected; the 21 "pyramidal"
cells are the kept cells without a label. The 21 tetrode cells may be the paper's kept cells,
but the data set does not say so.

## Shared measures for the timing claims

Cells: tetrode, kept, extracellular spike present, somatic (as defined above). The atypical cell
with a large initial positivity (Results) would be left out; the initial positivity before the
negative peak is printed for each cell. Readings, all run (claims file): O2 spikes (a) all,
(b) spontaneous (outside current steps); O3 (a) the channel with the largest averaged negative
peak, with (b) every channel reported; O4 intracellular start (a) dV/dt > 10 V/s, (b) maximal
second derivative; extracellular start (a) within 2 SD of baseline noise, (b) most negative
second derivative; intracellular end (thr) return to the threshold potential, (base) return to
the 2 ms pre-start baseline. Extracellular end is the late positive peak.

In [ ]:
def timing(fo, group, ch=None):
    w = hc1.cell_waves(R[fo]["files"], group, DROP)
    if w is None:
        return []
    im = hc1.ic_measures(w["ic"])
    best = T.loc[fo, "best"] if ch is None else ch
    best = int(best)
    em = hc1.ec_measures(w["ec"][best], w["noise"][best])
    out = []
    for ics in ("a", "b"):
        for ecs in ("a", "b"):
            for end in ("thr", "base"):
                ti0 = im[f"t_start_{ics}"]
                ti1 = im[f"t_end_{end}_{ics}"]
                te0 = em[f"t_start_{ecs}"]
                te1 = em["t_pos"]
                out.append(dict(folder=fo, spikes=group, channel=best, ic_start=ics, ec_start=ecs, ic_end=end, n=w["n"],
                                lead_start=ti0 - te0, lead_end=ti1 - te1, dur_ic=ti1 - ti0, dur_ec=te1 - te0,
                                latency=im["t_peak"] - em["t_neg"]))
    return out

timing_cells = T[(T.electrode == "tetrode") & T.kept & T.ec_present & (T.site == "somatic")].index.tolist()
print("somatic tetrode cells kept with an extracellular spike:", timing_cells)
print("initial positivity (uV) and negative peak (uV):")
print(T.loc[timing_cells, ["initial_pos_uV", "ec_neg_uV"]].round(1).to_string())
TM = pd.DataFrame([row for fo in timing_cells for g in ("all", "spont") for row in timing(fo, g)])
TM_ch = pd.DataFrame([row for fo in timing_cells for ch in range(int(T.loc[fo, "n_ec"])) for row in timing(fo, "all", ch)])
keys = ["spikes", "ic_start", "ec_start", "ic_end"]
summ = TM.groupby(keys).agg(n_cells=("folder", "nunique"), lead_start=("lead_start", "mean"), lead_start_sd=("lead_start", "std"),
                            lead_end=("lead_end", "mean"), lead_end_sd=("lead_end", "std"),
                            dur_ic=("dur_ic", "mean"), dur_ic_sd=("dur_ic", "std"),
                            dur_ec=("dur_ec", "mean"), dur_ec_sd=("dur_ec", "std"),
                            latency=("latency", "mean"), latency_sd=("latency", "std")).reset_index()

## fig1c-start-end-timing

> In the 11 somatic recordings, the start and the end of the average extracellular spike fall at
> almost the same times as those of the average intracellular action potential (0.01 +/- 0.02 ms
> and 0.02 +/- 0.14 ms earlier, mean +/- SD).

Reproduced when, under one reading, both means over cells are within tolerance (0.01 +/- 0.02
and 0.02 +/- 0.14 ms).

In [ ]:
print(TM[(TM.spikes == "all") & (TM.ic_start == "a") & (TM.ec_start == "a") & (TM.ic_end == "thr")].round(3).to_string())
s = summ.copy()
s["start_ok"] = (s.lead_start - 0.01).abs() <= 0.02
s["end_ok"] = (s.lead_end - 0.02).abs() <= 0.14
print(s[keys + ["n_cells", "lead_start", "lead_start_sd", "lead_end", "lead_end_sd", "start_ok", "end_ok"]].round(3).to_string())
print("readings with both within tolerance:", int((s.start_ok & s.end_ok).sum()), "of", len(s))
# O3 (b): every channel, reading all/a/a/thr
chs = TM_ch[(TM_ch.ic_start == "a") & (TM_ch.ec_start == "a") & (TM_ch.ic_end == "thr")]
print("per channel (O3 b), reading all spikes, starts (a), end at threshold:")
print(chs.pivot_table(index="folder", columns="channel", values=["lead_start", "lead_end"]).round(3).to_string())

**Result: not reproduced.** Eight somatic tetrode cells qualify (11 in the paper). Under none of
the 16 readings are both mean leads within tolerance: the mean start lead lies between -0.174
and +0.161 ms across readings, never within 0.01 +/- 0.02 ms, while the mean end lead is within
0.02 +/- 0.14 ms under 6 readings. The files are sampled every 0.05 or 0.1 ms, coarser than the
start's tolerance.

## ap-duration-match

> The action potential duration is the same whether measured from the intracellular waveform
> (1.28 +/- 0.18 ms) or from the extracellular one, start to late positive peak (1.27 +/- 0.19 ms).

In [ ]:
s["dic_ok"] = (s.dur_ic - 1.28).abs() <= 0.18
s["dec_ok"] = (s.dur_ec - 1.27).abs() <= 0.19
print(s[keys + ["n_cells", "dur_ic", "dur_ic_sd", "dur_ec", "dur_ec_sd", "dic_ok", "dec_ok"]].round(3).to_string())
print("readings with both within tolerance:", int((s.dic_ok & s.dec_ok).sum()), "of", len(s))
print(TM[(TM.spikes == "all") & (TM.ic_start == "a") & (TM.ec_start == "a")][["folder", "ic_end", "dur_ic", "dur_ec"]].round(3).to_string())

**Result: reproduced.** Under 6 of the 16 readings both mean durations are within tolerance; the
first in the criterion's order (all spikes, intracellular start at the maximal second
derivative, extracellular start within 2 SD, end at return to baseline) gives 1.382 ms
intracellular and 1.281 ms extracellular over 8 somatic cells. Under the paper's own end
definition with the dV/dt start (all spikes) the intracellular duration is 1.547 ms, outside
tolerance.

## duration-correlation

> Across somatic cells, intracellular and extracellular spike durations are linearly related
> (R = 0.66, P < 0.03).

Least-squares regression per reading; reproduced when R is positive, within 0.594 to 0.726, with
P < 0.03; partially when positive with P < 0.03 outside tolerance.

In [ ]:
rr = []
for k, g in TM.groupby(keys):
    g = g.dropna(subset=["dur_ic", "dur_ec"])
    if len(g) >= 3:
        lr = stats.linregress(g.dur_ic, g.dur_ec)
        rr.append(dict(zip(keys, k), n=len(g), R=lr.rvalue, p=lr.pvalue))
RR = pd.DataFrame(rr)
RR["R_in_tol"] = (RR.R - 0.66).abs() <= 0.066
RR["sig"] = (RR.R > 0) & (RR.p < 0.03)
print(RR.round(4).to_string())
print("readings reproduced:", int((RR.R_in_tol & RR.sig).sum()), "; positive with P < 0.03:", int(RR.sig.sum()), "of", len(RR))

**Result: partially reproduced.** The durations are positively related with P < 0.03 under 4 of
16 readings (R 0.759 to 0.945, all spikes), but no reading gives R within 0.594 to 0.726 with P
< 0.03; the one reading with R in tolerance (0.644) has P = 0.085. With spontaneous spikes only,
no reading reaches P < 0.03 (R 0.27 to 0.51). n = 8 cells.

## neg-peak-precedes-intra-peak

> In somatic recordings the negative peak of the extracellular spike precedes the peak of the
> intracellular action potential by 0.20 +/- 0.08 ms.

In [ ]:
lat = TM[(TM.ic_start == "a") & (TM.ec_start == "a") & (TM.ic_end == "thr")][["folder", "spikes", "latency"]]
print(lat.pivot(index="folder", columns="spikes", values="latency").round(3).to_string())
for g, x in lat.groupby("spikes"):
    print(f"{g}: mean {x.latency.mean():.3f} ms, SD {x.latency.std():.3f}, n {len(x)}; within 0.20 +/- 0.08: {abs(x.latency.mean() - 0.2) <= 0.08}")
latc = TM_ch[(TM_ch.ic_start == "a") & (TM_ch.ec_start == "a") & (TM_ch.ic_end == "thr")]
print("per channel (O3 b), all spikes:")
print(latc.pivot(index="folder", columns="channel", values="latency").round(3).to_string())

**Result: reproduced.** Over 8 somatic cells the extracellular negative peak precedes the
intracellular peak by 0.207 +/- 0.094 ms (all spikes) and 0.208 +/- 0.101 ms (spontaneous
spikes), within 0.20 +/- 0.08 ms and positive. The value is close on every tetrode channel of
each cell except the second tetrode of D83.

## fig1c-derivative-mismatch

> The extracellular spike resembles the negative first derivative of the intracellular action
> potential during the rising phase but not during the falling phase and afterhyperpolarization,
> where the derivative follows a much faster time course.

Per somatic cell (the cells above), readings (a) after scaling the negative derivative to the
extracellular negative peak, the Pearson correlation over the rising phase (intracellular start
to peak) exceeds that over the falling phase (peak to return to threshold); (b) the time from
the negative to the positive peak is shorter for the negative derivative than for the
extracellular spike. Reproduced when one reading holds in more than half of the cells.

In [ ]:
dm = []
for fo in timing_cells:
    for g in ("all", "spont"):
        w = hc1.cell_waves(R[fo]["files"], g, DROP)
        im = hc1.ic_measures(w["ic"])
        b = int(T.loc[fo, "best"])
        e = w["ec"][b]
        em = hc1.ec_measures(e, w["noise"][b])
        nd = -np.gradient(w["ic"], GRID[1] - GRID[0])
        scale = em["amp_neg"] / nd.min() * -1 if nd.min() < 0 else np.nan
        sd = em["base"] + nd * scale
        rise = (GRID >= im["t_start_a"]) & (GRID <= im["t_peak"])
        fall = (GRID > im["t_peak"]) & (GRID <= im["t_end_thr_a"])
        r_rise = stats.pearsonr(sd[rise], e[rise])[0]
        r_fall = stats.pearsonr(sd[fall], e[fall])[0]
        t_dneg = GRID[np.argmin(nd[(GRID > -1) & (GRID < 1)]) + np.flatnonzero((GRID > -1) & (GRID < 1))[0]]
        m2 = (GRID > t_dneg) & (GRID < t_dneg + 2.5)
        t_dpos = GRID[m2][np.argmax(nd[m2])]
        dm.append(dict(folder=fo, spikes=g, r_rise=r_rise, r_fall=r_fall, a_holds=r_rise > r_fall,
                       deriv_neg_to_pos=t_dpos - t_dneg, ec_neg_to_pos=em["t_pos"] - em["t_neg"],
                       b_holds=(t_dpos - t_dneg) < (em["t_pos"] - em["t_neg"])))
DM = pd.DataFrame(dm)
print(DM.round(3).to_string())
print(DM.groupby("spikes")[["a_holds", "b_holds"]].agg(["sum", "count"]).to_string())

In [ ]:
fo = timing_cells[0]
w = hc1.cell_waves(R[fo]["files"], "all", DROP)
b = int(T.loc[fo, "best"])
fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax[0].plot(GRID, w["ic"], "k")
ax[0].set_ylabel("intracellular (mV)")
for j in range(len(w["ec"])):
    ax[1].plot(GRID, w["ec"][j], lw=1 if j != b else 2)
nd = -np.gradient(w["ic"], GRID[1] - GRID[0])
ax[1].plot(GRID, nd / -nd.min() * hc1.ec_measures(w["ec"][b], w["noise"][b])["amp_neg"], "k--", label="-dV/dt scaled")
ax[1].set_ylabel("extracellular (uV)")
ax[1].set_xlabel("ms from AP peak")
ax[1].legend()
ax[1].set_xlim(-2, 4)
ax[0].set_title(f"{fo}, {w['n']} APs")
plt.show()
print(f"plotted range: intracellular {w['ic'].min():.1f} to {w['ic'].max():.1f} mV; extracellular "
      f"{w['ec'].min():.1f} to {w['ec'].max():.1f} uV over {GRID[0]} to {GRID[-1]} ms (axis shows -2 to 4 ms)")

**Result: reproduced.** Reading (b) holds in 7 of 8 somatic cells with either spike set: the
negative derivative goes from its negative to its positive peak faster than the extracellular
spike. Reading (a) holds in 6 of 8 (all spikes) and 5 of 8 (spontaneous): the correlation with
the scaled derivative is higher over the rising phase than over the falling phase.

## notch-and-ahp-landmarks

> A notch on the ascending phase of the extracellular spike after its negative peak coincides
> with the peak of the intracellular action potential, and a change of slope after the late
> positive peak coincides with the peak of the afterhyperpolarization.

Per pyramidal cell (tetrode, kept, extracellular spike present, somatic and dendritic), on the
averaged waveforms of the largest channel: the notch is a local minimum of the extracellular
first derivative between the negative peak and the late positive peak; the slope change is a
local maximum of the second derivative within 3 ms after the late positive peak; the AHP peak is
the intracellular minimum within 5 ms after the return to threshold. A local extreme counts when
its prominence is at least 5% of the largest absolute value of that derivative in the window, a
standard choice that the paper leaves open (shown at 2% and 10% too). Agreement windows (a)
0.1 ms, (b) 0.2 ms.

In [ ]:
from scipy.signal import find_peaks

pyr_cells = T[(T.electrode == "tetrode") & T.kept & T.ec_present].index.tolist()

def landmarks(fo, group, prom):
    w = hc1.cell_waves(R[fo]["files"], group, DROP)
    im = hc1.ic_measures(w["ic"])
    b = int(T.loc[fo, "best"])
    em = hc1.ec_measures(w["ec"][b], w["noise"][b])
    d1, d2, ni, pi = em["d1"], em["d2"], em["ni"], em["pi"]
    out = dict(folder=fo, spikes=group, prom=prom, site=T.loc[fo, "site"], t_ic_peak=im["t_peak"], t_ahp=im["t_ahp"])
    seg = d1[ni + 1:pi]
    notch = np.nan
    if len(seg) > 3:
        pk, _ = find_peaks(-seg, prominence=prom * np.abs(seg).max())
        if len(pk):
            notch = GRID[ni + 1 + pk[0]]
    out["t_notch"] = notch
    m = (GRID > em["t_pos"]) & (GRID <= em["t_pos"] + 3)
    seg2 = d2[m]
    sc = np.nan
    if m.sum() > 3:
        pk2, _ = find_peaks(seg2, prominence=prom * np.abs(seg2).max())
        if len(pk2):
            sc = GRID[m][pk2[0]]
    out["t_slope_change"] = sc
    # variant, not the criterion: the first local minimum of the intracellular trace within 3 ms
    # after the return to threshold (a fast AHP), against the same slope change
    e = im["t_end_thr_a"]
    m3 = (GRID >= e) & (GRID <= e + 3) if np.isfinite(e) else np.zeros(len(GRID), bool)
    tf = np.nan
    if m3.sum() > 3:
        pk3, _ = find_peaks(-w["ic"][m3])
        if len(pk3):
            tf = GRID[m3][pk3[0]]
    out["t_fast_ahp"] = tf
    return out

LM = pd.DataFrame([landmarks(fo, g, p) for fo in pyr_cells for g in ("all", "spont") for p in (0.02, 0.05, 0.10)])
LM["d_notch"] = LM.t_notch - LM.t_ic_peak
LM["d_slope"] = LM.t_slope_change - LM.t_ahp
print(LM[LM.prom == 0.05].round(3).to_string())
res = []
for (g, p), x in LM.groupby(["spikes", "prom"]):
    for win in (0.1, 0.2):
        hn = x.t_notch.notna()
        hs = x.t_slope_change.notna()
        n_ok = int((x.d_notch.abs() <= win).sum())
        s_ok = int((x.d_slope.abs() <= win).sum())
        res.append(dict(spikes=g, prom=p, window=win, cells=len(x), with_notch=int(hn.sum()), notch_agrees=n_ok,
                        with_slope_change=int(hs.sum()), slope_agrees=s_ok,
                        notch_majority=n_ok > hn.sum() / 2, slope_majority=s_ok > hs.sum() / 2))
LR = pd.DataFrame(res)
print(LR.to_string())
LM["d_slope_fast"] = LM.t_slope_change - LM.t_fast_ahp
v = LM[(LM.prom == 0.05)]
for g, x in v.groupby("spikes"):
    hs = x.d_slope_fast.notna()
    print(f"variant, fast AHP ({g} spikes): cells with both landmarks {int(hs.sum())}; slope change within 0.1 ms: "
          f"{int((x.d_slope_fast.abs() <= 0.1).sum())}, within 0.2 ms: {int((x.d_slope_fast.abs() <= 0.2).sum())}; "
          f"median difference {x.d_slope_fast.median():.3f} ms")

**Result: not reproduced.** Of 20 pyramidal cells, a notch is found in 9 (at 5% prominence; 4 to
11 across 2% to 10%) and agrees with the intracellular peak within 0.2 ms in 3 or 4 of them, at
most 4 under any reading, never more than half. A slope change after the late positive peak is
found in all 20 cells but falls within 0.2 ms of the AHP peak, as the criterion defines it
(minimum within 5 ms after the return to threshold), in at most 1. The variant with the first
fast AHP trough gives 0 or 1 of 12 or 13 within 0.2 ms.

## fig3b-latency-vs-amplitude

> The latency from the extracellular negative peak to the intracellular peak grows as the
> intracellular action potential amplitude falls, longer in dendritic than in somatic recordings.

Per pyramidal cell (tetrode, kept, extracellular spike present; somatic and dendritic), latency
against AP amplitude, O6 (a) peak minus threshold, (b) peak minus the 2 ms pre-start baseline;
spikes O2 (a) all, (b) spontaneous; (a) Pearson, (b) Spearman. Reproduced when, under one
reading, the correlation is negative with two-sided P < 0.05 and the dendritic median latency
exceeds the somatic one.

In [ ]:
def lat_amp(fo, g):
    w = hc1.cell_waves(R[fo]["files"], g, DROP)
    im = hc1.ic_measures(w["ic"])
    b = int(T.loc[fo, "best"])
    em = hc1.ec_measures(w["ec"][b], w["noise"][b])
    return dict(folder=fo, spikes=g, site=T.loc[fo, "site"], latency=im["t_peak"] - em["t_neg"],
                amp_a=im["amp_thr_a"], amp_b=im["amp_base_a"], peak=im["peak"])

LA = pd.DataFrame([lat_amp(fo, g) for fo in pyr_cells for g in ("all", "spont")])
print(LA.round(3).to_string())
r3 = []
for g, x in LA.groupby("spikes"):
    for amp in ("amp_a", "amp_b"):
        y = x.dropna(subset=[amp])
        pr = stats.pearsonr(y[amp], y.latency)
        sr_ = stats.spearmanr(y[amp], y.latency)
        med_d = y[y.site == "dendritic"].latency.median()
        med_s = y[y.site == "somatic"].latency.median()
        for name, (rv, pv) in (("pearson", pr), ("spearman", sr_)):
            r3.append(dict(spikes=g, amplitude=amp, test=name, n=len(y), r=rv, p=pv,
                           median_lat_dendritic=med_d, median_lat_somatic=med_s,
                           holds=(rv < 0) and (pv < 0.05) and (med_d > med_s)))
R3 = pd.DataFrame(r3)
print(R3.round(4).to_string())
print("counts by site:", LA[LA.spikes == "all"].site.value_counts().to_dict())

In [ ]:
x = LA[LA.spikes == "all"]
fig, ax = plt.subplots(figsize=(5, 4))
for site, c in (("somatic", "k"), ("dendritic", "0.6")):
    y = x[x.site == site]
    ax.scatter(y.amp_a, y.latency, c=c, label=site)
ax.set_xlabel("AP amplitude, peak minus threshold (mV)")
ax.set_ylabel("latency, extracellular negative peak to AP peak (ms)")
ax.legend()
plt.show()
print(f"plotted: amplitude {x.amp_a.min():.1f} to {x.amp_a.max():.1f} mV, latency {x.latency.min():.3f} to {x.latency.max():.3f} ms; all points shown")

**Result: reproduced.** Over 20 pyramidal cells (8 somatic, 12 dendritic) latency falls with AP
amplitude under every reading (Pearson r -0.698 to -0.712, P 0.0004 to 0.0006; Spearman -0.529
to -0.592, P 0.006 to 0.016), and the dendritic median latency exceeds the somatic one (0.295
against 0.2025 ms, all spikes).

## fig3a-example-pair

> In two cells impaled on the same track, a dendrite 240 um deeper than a soma, the
> extracellular peak led the intracellular peak by 0.55 ms in the dendritic recording and by 0.28
> ms in the somatic one.

The pair is two cells recorded one after the other with the same electrodes, the first somatic,
the second dendritic (Fig. 3A legend). The data set has no depth for these cells; the folder
names (dAAATC: animal AAA, track T, cell C, an inference from the names, which the descriptor
does not explain) give the candidates: consecutive cells of one animal and track.

In [ ]:
cand = []
tet_names = sorted(T[T.electrode == "tetrode"].index, key=lambda f: (hc1.animal(f), f))
for a_ in tet_names:
    for b_ in tet_names:
        if a_ != b_ and a_[:-1] == b_[:-1] and int(b_[-1]) == int(a_[-1]) + 1:
            for g in ("all", "spont"):
                la = LA.set_index(["folder", "spikes"])
                if (a_, g) in la.index and (b_, g) in la.index:
                    cand.append(dict(first=a_, second=b_, spikes=g, site_first=T.loc[a_, "site"], site_second=T.loc[b_, "site"],
                                     latency_first=la.loc[(a_, g), "latency"], latency_second=la.loc[(b_, g), "latency"]))
CA = pd.DataFrame(cand)
CA["fits_order"] = (CA.site_first == "somatic") & (CA.site_second == "dendritic")
CA["somatic_ok"] = (CA.latency_first - 0.28).abs() <= 0.028
CA["dendritic_ok"] = (CA.latency_second - 0.55).abs() <= 0.055
print(CA.round(3).to_string())

**Result: blocked (metadata insufficient).** The data set has no depth or track record for these
cells, so the pair cannot be identified. The folder names allow two candidates in which a
somatic cell is followed by a dendritic one: D11221 then D11222 (latencies 0.250 and 0.975 ms)
and D13521 then D13522 (0.210 and 0.245 ms). Neither matches 0.28 and 0.55 ms within tolerance.

## fig3c-backprop-velocity

> Converting intracellular action potential amplitude to distance from the soma (4.9 um/mV, from
> Kamondi et al. 1998), the latencies give an average back-propagation velocity of 1.61 m/s.

Distance = 4.9 um/mV times the amplitude deficit. Reference amplitude (a) mean of the somatic
cells, (b) largest amplitude. Velocity (a) inverse slope of latency on distance, all pyramidal
cells; (b) slope of distance on latency; (c) mean over dendritic cells of distance over latency in
excess of the mean somatic latency. O6 and O2 readings as above. 1 um/ms is 0.001 m/s.

In [ ]:
vel = []
for (g), x in LA.groupby("spikes"):
    for amp in ("amp_a", "amp_b"):
        y = x.dropna(subset=[amp])
        soma = y[y.site == "somatic"]
        for refname, ref in (("somatic mean", soma[amp].mean()), ("largest", y[amp].max())):
            dist = 4.9 * (ref - y[amp])  # um
            l1 = stats.linregress(dist, y.latency)
            l2 = stats.linregress(y.latency, dist)
            den = y.site == "dendritic"
            exc = y.latency[den] - soma.latency.mean()
            vc = np.mean(dist[den] / exc) / 1000
            vel.append(dict(spikes=g, amplitude=amp, reference=refname, v_a=1 / l1.slope / 1000 if l1.slope != 0 else np.nan,
                            v_b=l2.slope / 1000, v_c=vc, n=len(y), n_dendritic=int(den.sum())))
V = pd.DataFrame(vel)
for c in ("v_a", "v_b", "v_c"):
    V[c + "_ok"] = (V[c] - 1.61).abs() <= 0.161
print(V.round(3).to_string())
print("readings within 1.61 +/- 0.161 m/s:", int(V[["v_a_ok", "v_b_ok", "v_c_ok"]].values.sum()), "of", 3 * len(V))

**Result: not reproduced.** No reading gives a velocity within 1.61 +/- 0.161 m/s (0 of 24).
With all spikes the readings give 0.23 to 0.49 m/s; with spontaneous spikes reading (c) gives
0.79 to 2.46 m/s depending on the reference amplitude, and the other two readings give 0.24 to
0.49 m/s.

## fig4bc-burst-attenuation

> During current-evoked trains the intracellular action potential becomes smaller and slower,
> through depolarization of the baseline with the peak unchanged, and the extracellular spike
> follows: its amplitude and the slope of its positive-going phase decrease and the notch fades.

Cells with depolarizing current steps that evoke trains: here, kept cells with an extracellular
spike that have at least 10 steps reaching a 5th AP (a threshold chosen here; the paper's Fig. 4B
averages 176 to 243 trains). Per cell, waveforms averaged by order in the train, 1st to 5th.
Readings for "decrease": (a) 5th below 1st, (b) falling from each AP to the next from the 2nd on.
Reproduced when, in more than half of the cells, the intracellular amplitude, maximal rising and
falling slopes, extracellular amplitude and maximal slope of the positive-going phase all
decrease, the pre-spike baseline rises, and the peak changes by less than the baseline.
Amplitudes O6 (a, b) and O7 (a, b). Tetrode cells are the main set (the paper's Fig. 4 cells were
tetrode recordings); the silicon-probe cells are shown as a variant. The notch is not scored.

In [ ]:
def train_measures(fo):
    rows = []
    for k in range(1, 6):
        w = hc1.cell_waves(R[fo]["files"], f"ord{k}", DROP)
        if w is None:
            return None
        im = hc1.ic_measures(w["ic"])
        b = int(T.loc[fo, "best"])
        em = hc1.ec_measures(w["ec"][b], w["noise"][b])
        rows.append(dict(folder=fo, order=k, n=w["n"], ic_amp_a=im["amp_thr_a"], ic_amp_b=im["amp_base_a"],
                         rise=im["rise_max"], fall=im["fall_max"], baseline=im["base_a"], peak=im["peak"],
                         ec_amp_a=em["amp_np"], ec_amp_b=em["amp_neg"], ec_pos_slope=em["rise_slope"]))
    return pd.DataFrame(rows)

def decreases(v, reading):
    v = np.asarray(v, float)
    return bool(v[4] < v[0]) if reading == "a" else bool(np.all(np.diff(v[1:]) < 0))

n5 = {fo: sum(f["sums"]["ord5"][0] for f in R[fo]["files"] if "sums" in f and f["member"] not in DROP) for fo in T.index}
T["trains_to_5th"] = pd.Series(n5)
train_cells = T[T.kept & T.ec_present & (T.trains_to_5th >= 10)].index.tolist()
print("cells with >= 10 trains reaching a 5th AP:", {fo: int(T.loc[fo, "trains_to_5th"]) for fo in train_cells})
TR = {fo: train_measures(fo) for fo in train_cells}
TR = {k: v for k, v in TR.items() if v is not None}
out = []
for fo, x in TR.items():
    for rd in ("a", "b"):
        for o6 in ("a", "b"):
            for o7 in ("a", "b"):
                ica = x[f"ic_amp_{o6}"].values
                eca = x[f"ec_amp_{o7}"].values
                base = x.baseline.values
                pk = x.peak.values
                i0 = 0 if rd == "a" else 1
                parts = dict(ic_amp=decreases(ica, rd), rise=decreases(x.rise, rd), fall=decreases(x.fall, rd),
                             ec_amp=decreases(eca, rd), ec_slope=decreases(x.ec_pos_slope, rd),
                             base_rises=decreases(-base, rd),
                             peak_less_than_base=abs(pk[4] - pk[i0]) < abs(base[4] - base[i0]))
                out.append(dict(folder=fo, electrode=T.loc[fo, "electrode"], reading=rd, O6=o6, O7=o7, **parts,
                                amplitude_parts=parts["ic_amp"] and parts["ec_amp"], all_parts=all(parts.values())))
FB = pd.DataFrame(out)
for fo, x in TR.items():
    print(fo, T.loc[fo, "electrode"])
    print(x.round(3).to_string(index=False))
print(FB.to_string())
agg = FB.groupby(["electrode", "reading", "O6", "O7"]).agg(cells=("folder", "count"), all_parts=("all_parts", "sum"),
                                                           amplitude_parts=("amplitude_parts", "sum")).reset_index()
print(agg.to_string())

Variant: the same test with the selection threshold raised to 20 and 50 trains reaching a 5th AP.

In [ ]:
for thr_n in (10, 20, 50):
    sel = [fo for fo in TR if T.loc[fo, "trains_to_5th"] >= thr_n]
    x = FB[FB.folder.isin(sel) & (FB.electrode == "tetrode")]
    a = x.groupby(["reading", "O6", "O7"]).agg(cells=("folder", "count"), all_parts=("all_parts", "sum")).reset_index()
    a["majority"] = a.all_parts > a.cells / 2
    print(f"tetrode cells with >= {thr_n} trains:")
    print(a.to_string())

**Result: reproduced, resting on the assumption about which cells form the group.** Among the 17
tetrode cells with at least 10 trains reaching a 5th AP, all parts hold in 9 under reading (a)
with the extracellular amplitude measured from baseline to negative peak (O7 b), just over half;
with the negative-to-positive amplitude (O7 a) they hold in 6, and under reading (b) in at most
2. The outcome depends on the selection: with at least 20 trains, 6 of 13 (not a majority); with
at least 50, 5 of 8. The amplitude parts alone hold in 12 of 17 (O7 b). In the silicon-probe
cells all parts hold in 1 of 7.

## fig4d-amplitude-correlation

> Within high-frequency evoked bursts (about 200 Hz) in one cell, extracellular spike amplitude
> correlates positively with intracellular action potential amplitude (R = 0.40, P < 0.0001).

Spikes inside depolarizing steps with an interspike interval below 5 ms to a neighbour; per
spike, intracellular amplitude O6 (a) peak minus threshold, (b) peak minus the mean of -3 to
-1.5 ms; extracellular amplitude O7 (a) negative to positive, (b) baseline to negative peak, on the
cell's largest channel. Per candidate cell with at least 30 such spikes, a least-squares
regression of extracellular on intracellular amplitude. Reproduced when some cell gives R within
0.36 to 0.44 and P < 0.0001.

In [ ]:
reg = []
for fo in T[T.kept & T.ec_present].index:
    b = int(T.loc[fo, "best"])
    ica_, icb_, eca_, ecb_ = [], [], [], []
    for f in R[fo]["files"]:
        if f["n_spikes"] == 0 or f["member"] in DROP or len(f["ec"]) != T.loc[fo, "n_ec"]:
            continue
        sp = f["spikes"]
        m = sp["in_dep"] & ((sp["isi_prev"] < 5) | (sp["isi_next"] < 5))
        si, se = f["spike_ic"], f["spike_ec"]
        ica_.append((si["peak"] - si["thr"])[m]); icb_.append((si["peak"] - si["base"])[m])
        eca_.append(se["amp_np"][m, b]); ecb_.append(se["amp_neg"][m, b])
    if not ica_:
        continue
    ica_, icb_, eca_, ecb_ = map(np.concatenate, (ica_, icb_, eca_, ecb_))
    if len(icb_) < 30:
        continue
    for o6, xi in (("a", ica_), ("b", icb_)):
        for o7, ye in (("a", eca_), ("b", ecb_)):
            ok = np.isfinite(xi) & np.isfinite(ye)
            if ok.sum() < 30:
                continue
            lr = stats.linregress(xi[ok], ye[ok])
            reg.append(dict(folder=fo, electrode=T.loc[fo, "electrode"], O6=o6, O7=o7, n=int(ok.sum()), R=lr.rvalue, p=lr.pvalue,
                            slope_uV_per_mV=lr.slope))
RG = pd.DataFrame(reg)
RG["in_tol"] = (RG.R - 0.40).abs() <= 0.04
RG["p_ok"] = RG.p < 0.0001
print(RG.round(5).to_string())
print("cells:", RG.folder.nunique(), "; rows with R in 0.36-0.44 and P < 0.0001:",
      RG[RG.in_tol & RG.p_ok][["folder", "O6", "O7", "R"]].round(3).values.tolist())
print("rows with R > 0 and P < 0.0001:", int(((RG.R > 0) & RG.p_ok).sum()), "of", len(RG))

**Result: reproduced.** Two cells give R within 0.36 to 0.44 with P < 0.0001: D15121 (O6 a, O7
a; R = 0.367, n = 514) and D12821 (O6 a, O7 b; R = 0.431, n = 95). The relationship is not
general: of 54 cell and reading rows over 16 cells, 10 are positive with P < 0.0001, and D11222,
D13522 and D17111 give negative correlations.

## fig7-vertical-spread

> On a linear silicon probe, the spike of a cell can be recorded 100 um or more above and below
> the site nearest the soma (a vertical span of at least 200 um), and its rise and decay slopes
> decrease with distance from the soma along with its amplitude.

Silicon-probe cells (kept, extracellular spike present; D17212, labelled an interneuron in the
spreadsheet, left out), averaged on the intracellular APs, on the band as recorded (1 Hz to 5
kHz, descriptor; the Fig. 7 legend's 1 Hz to 10 kHz is wider than the files hold). Site order
from the spreadsheet column "hextrode chans" (site/channel pairs for one shank; the descriptor:
"the remaining channels form another shank in a similar way", taken as the same list shifted by
one channel where the shifted list is exactly the remaining channels). For the files with six
extracellular channels (D172 to D189) the column lists channels up to 10, which do not exist in
them; the order used is the listed channel divided by two (the six channels taken as the even
channels of the twelve-channel numbering), an assumption checked below by the smoothness of the
amplitude profile, with the result shown without these cells. Site spacing 25 um (Results). The
site with the largest spike is the soma site. Present: averaged negative peak above 2 SD of the
averaged noise. Reproduced when, in more than half of the cells, the spike is present 100 um
(four sites) or more from the soma site on at least one side, and amplitude, maximal rise slope
(positive-going phase) and maximal decay slope (negative-going phase) fall with distance from
the soma site (Spearman correlation over the six sites below 0).

In [ ]:
def shank_orders(fo, n_ec):
    if fo not in lab.index:
        return None, "not in spreadsheet"
    pairs = re.findall(r"(\d)\s*/\s*(\d+)", str(lab.loc[fo, "hextrode chans"]))
    order = [int(c) for _, c in sorted(pairs, key=lambda p: int(p[0]))]
    if n_ec == 12:
        shanks = [order]
        rest = sorted(set(range(12)) - set(order))
        for s in (1, -1):
            o2 = [c + s for c in order]
            if sorted(o2) == rest:
                shanks.append(o2)
        return shanks, "listed" + (" + shifted" if len(shanks) > 1 else " only")
    if n_ec == 6:
        return [[c // 2 for c in order]], "assumed (channel / 2)"
    return None, "unknown"

def vertical(fo, group="all_raw"):
    w = hc1.cell_waves(R[fo]["files"], group, DROP)
    n_ec = len(w["ec"])
    shanks, how = shank_orders(fo, n_ec)
    if shanks is None:
        return None
    ms = [hc1.ec_measures(w["ec"][j], w["noise"][j]) for j in range(n_ec)]
    amp = np.array([m["amp_neg"] for m in ms])
    best_shank = max(shanks, key=lambda s: amp[s].max())
    if amp[best_shank].max() < amp.max():
        return dict(folder=fo, order=how, note="largest spike on a shank of unknown order")
    a = amp[best_shank]
    snr = a / w["noise"][best_shank]
    soma = int(np.argmax(a))
    dist = np.abs(np.arange(6) - soma)
    present = snr > 2
    far = int(dist[present].max())
    rise = np.array([ms[c]["rise_slope"] for c in best_shank])
    decay = np.array([ms[c]["fall_slope"] for c in best_shank])
    tneg = np.array([ms[c]["t_neg"] for c in best_shank])
    rho = {k: stats.spearmanr(dist, v)[0] for k, v in (("amp", a), ("rise", rise), ("decay", decay))}
    return dict(folder=fo, order=how, shank=best_shank, soma_site=soma + 1, amp_uV=np.round(a, 1).tolist(),
                snr=np.round(snr, 1).tolist(), farthest_present_sites=far, farthest_um=25 * far,
                spread_100=far >= 4, rho_amp=rho["amp"], rho_rise=rho["rise"], rho_decay=rho["decay"],
                slopes_fall=all(v < 0 for v in rho.values()), t_neg=np.round(tneg, 3).tolist(), present=present.tolist())

si_cells = [fo for fo in T[(T.electrode == "silicon probe") & T.kept & T.ec_present].index if fo != "d17212"]
V7 = pd.DataFrame([v for v in (vertical(fo) for fo in si_cells) if v is not None])
print("silicon-probe cells considered:", si_cells)
print(V7.drop(columns=["t_neg"]).round(3).to_string())
ok7 = V7.dropna(subset=["spread_100"])
ok7 = ok7.assign(holds=ok7.spread_100.astype(bool) & ok7.slopes_fall.astype(bool))
for name, x in (("all cells with a known order", ok7), ("without the assumed six-channel order", ok7[~ok7.order.str.startswith("assumed")])):
    print(f"{name}: {len(x)} cells; present >= 100 um: {int(x.spread_100.sum())}; amplitude and slopes fall: "
          f"{int(x.slopes_fall.sum())}; both: {int(x.holds.sum())}; majority: {x.holds.sum() > len(x) / 2}")

**Result: not reproduced.** Of 10 silicon-probe pyramidal cells with a site order, the spike is
present 100 um or more from the soma site in 3 (1 of the 4 whose order the spreadsheet lists),
not more than half. Amplitude and both slopes fall with distance from the soma site in 9 of 10
(4 of 4), so the second part of the claim holds.

## propagation-velocity-range

> Propagation velocities from the delay between extracellular negative peaks on sites along the
> somatodendritic axis range from 0.4 to 2 m/s in pyramidal cells (n = 25) and 0.5 to 1.5 m/s in
> pyramidal-layer interneurons (n = 6).

Reading (a) of the criterion, the anesthetized silicon-probe recordings, is the one the data set
allows; reading (b), the behaving-animal recordings, is not in it. Per cell (the cells of
fig7-vertical-spread, and D17212 as the interneuron), on the soma's shank and the sites where the
spike is present: pairing (a) the slope of a least-squares fit of negative-peak time on distance
from the soma site, (b) the soma site against the farthest site with the spike. Velocity is
distance over delay (1 um/ms = 0.001 m/s); a zero or negative delay gives no velocity in range.
Reproduced when every unit lies within 0.36 to 2.2 m/s (pyramidal) and 0.45 to 1.65 m/s
(interneurons).

In [ ]:
pv = []
for fo in si_cells + ["d17212"]:
    v = vertical(fo)
    if v is None or "shank" not in v or not isinstance(v.get("shank"), list):
        continue
    tn = np.array(v["t_neg"])
    pres = np.array(v["present"])
    soma = v["soma_site"] - 1
    d_um = 25 * np.abs(np.arange(6) - soma)
    m = pres
    va = vb = np.nan
    if m.sum() >= 2:
        lr = stats.linregress(d_um[m], tn[m])
        va = (1 / lr.slope) / 1000 if lr.slope > 0 else -np.inf
        far = np.flatnonzero(m)[np.argmax(d_um[m])]
        delay = tn[far] - tn[soma]
        vb = (d_um[far] / delay) / 1000 if delay > 0 else -np.inf
    kind = "interneuron" if fo == "d17212" else "pyramidal"
    lo, hi = (0.36, 2.2) if kind == "pyramidal" else (0.45, 1.65)
    pv.append(dict(folder=fo, kind=kind, order=v["order"], sites_present=int(m.sum()), v_fit=va, v_far=vb,
                   fit_in_range=lo <= va <= hi, far_in_range=lo <= vb <= hi, t_neg=v["t_neg"]))
PV = pd.DataFrame(pv)
print(PV.round(3).to_string())
for k, x in PV.groupby("kind"):
    print(f"{k}: {len(x)} cells; in range, fit pairing: {int(x.fit_in_range.sum())}; farthest-site pairing: {int(x.far_in_range.sum())}")
fin = PV.replace([np.inf, -np.inf], np.nan)
print("finite velocities, fit pairing (m/s):", fin.v_fit.dropna().round(2).tolist())
print("finite velocities, farthest-site pairing (m/s):", fin.v_far.dropna().round(2).tolist())

**Result: blocked (data missing), with reading (a) shown.** The counts the paper gives (25
pyramidal cells, 6 interneurons) exceed the anesthetized silicon-probe recordings in the data
set (18 folders, one labelled interneuron), so they come from the behaving-animal recordings,
which the data set does not hold. On reading (a), 5 of 10 pyramidal cells give a finite velocity
by the fit pairing (0.18 to 0.67 m/s), 3 of them within 0.36 to 2.2 m/s, and the other 5 a zero
or negative delay (shown as -inf); the interneuron D17212 gives 0.64 m/s under the assumed site
order.

## Blocked claims

- fig2-interneuron-narrow: the basket cell of Fig. 2 is a tetrode recording. The only cell the
  data set labels an interneuron (spreadsheet column "Interneuron") is D17212, a silicon-probe
  recording; none of the 21 tetrode folders has a cell type, and 19 of them are not in the
  spreadsheet at all. Picking the interneuron by its narrow spike would assume the finding.
  Blocked, `metadata_insufficient`.
- fig5-theta-smaller-spikes, theta-higher-rates, fig8-neighbouring-shanks: these use the
  behaving-animal sleep recordings of Csicsvari et al. 1999a (Methods). The data set holds only
  the anesthetized recordings (its title, README and the files listed above). Blocked,
  `data_missing`.
- fig6-amplitude-vs-distance: needs the histological tetrode-to-soma distance of 19 cells. No
  file, spreadsheet column or README holds a distance (the spreadsheet's `depth` is the electrode
  depth of 18 cells, 16 of them silicon-probe cells). Blocked, `data_missing`.
- tetrode-cell-count-estimate and silent-cells-inference depend on fig6-amplitude-vs-distance
  (directly or through it) and are blocked for the same reason.
- propagation-velocity-range: see its section; the paper's counts (25 pyramidal cells, 6
  interneurons) exceed what the anesthetized recordings hold, and the data set has none of the
  behaving-animal recordings.

## Summary

| Claim | Outcome |
|---|---|
| sample-cells | not reproduced (21 tetrode cells, not 33; no cell type labels) |
| fig1c-start-end-timing | not reproduced (start lead outside 0.01 +/- 0.02 ms in every reading) |
| ap-duration-match | reproduced (6 of 16 readings) |
| duration-correlation | partially reproduced (R 0.76 to 0.94, P < 0.03; none within 0.66 +/- 0.066) |
| neg-peak-precedes-intra-peak | reproduced (0.207 ms) |
| fig1c-derivative-mismatch | reproduced (7 of 8 cells) |
| notch-and-ahp-landmarks | not reproduced |
| fig2-interneuron-narrow | blocked: metadata insufficient |
| fig3b-latency-vs-amplitude | reproduced |
| fig3a-example-pair | blocked: metadata insufficient |
| fig3c-backprop-velocity | not reproduced (0.23 to 0.49 m/s with all spikes) |
| fig4bc-burst-attenuation | reproduced, resting on an assumption (9 of 17 cells) |
| fig4d-amplitude-correlation | reproduced (two cells) |
| fig5-theta-smaller-spikes | blocked: data missing |
| theta-higher-rates | blocked: data missing |
| fig6-amplitude-vs-distance | blocked: data missing |
| tetrode-cell-count-estimate | blocked: data missing |
| silent-cells-inference | blocked: data missing |
| fig7-vertical-spread | not reproduced (3 of 10 cells reach 100 um) |
| propagation-velocity-range | blocked: data missing |
| fig8-neighbouring-shanks | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The data set holds 21 tetrode cells where the paper
analysed 22, and gives no cell type, no somatic or dendritic site and no distance for them; the
site is classified here by the paper's rule from the averaged AP, which gives 8 somatic and 13
dendritic tetrode cells against the paper's 11 and 9. The channel holding the membrane
potential is chosen from the data in each file, and the intracellular scale assumes the
descriptor's typical gain of 50 (checked against the spreadsheet's resting potentials for 17
cells, within 5 mV for 11). The spike detection threshold, the windows used to find landmarks
on the averaged waveforms, the 3 kHz low-pass filter, the threshold of 10 trains for Fig. 4 and
the site order of the six-channel silicon-probe files are choices made here (see `hc1.py` and
the sections above). Averages pool all files of a cell, recorded at 10 or 20 kHz.

What was found about the data set. Only 5 of 39 folders have a README naming the channels, and
the intracellular channel moves between files of one cell (channel 5 or 6 in D112); two binary
formats (12-bit and 16-bit) are mixed without a note. The spreadsheet IntraExtra.xls covers 18
of the 39 cells, none of the 19 earliest tetrode cells, so the paper's cell types, impalement
sites and tetrode-to-soma distances are not in the data set. Two files of D81 (d813101.dat,
d813102.dat) repeat parts of d081fre.001.dat. Two files (d15611.001.dat, d16811.001.dat) have no
xml, and d15611 holds no intracellular APs. The spreadsheet's "hextrode chans" lists channels up
to 10 for cells whose files have six extracellular channels. The behaving-animal recordings of
Figures 5 and 8 are not in the data set.